### Methods for Test-Retest Reliability

##### These methods that are commonly used in tandem with PFM data were established in Laumann et al. 2015, Gordon et al. 2017, and others. This is a simplified script for demonstration purposes that runs on resting-state data from one child (9 year old boy). 

##### Individual-level functional connectivity estimates highly depend on the amount of data available. In this section, we will explore how connectivity estimates change as additional data are included and why collecting sufficient data is particularly important for precision functional mapping. 

##### We will also explore how changing different aspects of the reliability method (iterations, motion thresholding in QA section) also impact reliability estimates.

##### The goal is to move beyond thinking about reliability as a single number and instead directly observe how an individual's functional connectivity estimates stabilize as more high-quality data are added and are affected by how it is calculated. 

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

/Users/sarahechang/Desktop/research/presentations/precisionfMRIFLUX2026/01_Reliability


**Step 1. Load necessary packages.**

In [ ]:
import argparse,glob,json,logging,math,os,random,sys,time,warnings
import nibabel as nib
import numpy as np
import pandas as pd
import scipy.io
from IPython.display import SVG, display

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.cm import Set2, tab20
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patheffects as PathEffects
## suppress matplotlib warnings
mpl_logger = logging.getLogger('matplotlib')
mpl_logger.setLevel(logging.WARNING)

from itertools import cycle
from datetime import datetime

**Step 2. Configure parameters for analysis and for plotting.**

In [ ]:
# paths
sublist_path = PROJECT_ROOT / "scripts" / "sublist.txt"
data_dir = PROJECT_ROOT / "cPFM"
outdir = PROJECT_ROOT / "figures"

# parameters
subid = "cPFM08"
task="rest"
fd_thresh = 0.2
TR = 1.1

#plotting parameters for later
subids = []
studies = []
grouping = []
plot_groups = []
grouping_dict = {}

with open(sublist_path, 'r') as file:
    entries = [line.rstrip('\n') for line in file]
    for x in entries:
        theid = x.split()[0]
        subids.append(theid)
        thestudy = x.split()[1]
        studies.append(thestudy)
        thegroup = x.split()[2]
        grouping.append(thegroup)
        if thegroup != '999':
            if thegroup not in grouping_dict:
                grouping_dict[thegroup] = [theid]
                plot_groups.append(thestudy)
            else:
                grouping_dict[thegroup].append(theid)

**Step 3. Load reliability functions in scripts folder.**  

In [ ]:
from scripts.RestReliability_Flux2026 import (
    create_raw_data,
    create_true,
    plot_corrs,
    get_corrs,
    clean_dict,
    plot_final
)

**Step 4. This section contains several functions that do the following:**
(1) Motion censor and concatenate data 
(2) Randomly divide the concatenated data into a high confidence and test subsets. Calculate the Pearson's correlation between high confidence and test subsets. Iterate this process 1000 times.
(3) Plot the correlation values. The figure will be in the .../01_Reliability/figures folder.

In [ ]:
# Step 4.1: motion censor and concatenate Gordon parcel timeseries data
num_of_sessions=8
raw_concatenated_data = create_raw_data(subid, data_dir, task, fd_thresh, TR, num_of_sessions)

In [ ]:
display(SVG(filename='/Users/sarahechang/Desktop/research/presentations/precisionfMRIFLUX2026/notebooks/cPFM_reliability.svg'))

In [ ]:
# Step 4.2: Divide the concatenated data into a high confidence and test subsets. 
#Calculate correlation between each subset. Iterate 1000 times.

#define the length of the 'high confidence' subset in minutes.
truetime=15 
all_data_dict = {'all_times': []}
all_data_dict[subid] = []
update_thresh=50
rands=1000

print(f'\nRunning randomized iterations. (terminal update every {update_thresh})')
for i in range(1, rands + 1):
    if i % update_thresh == 0:
        print(f' -Iteration: {i}')
    hour_array, test_data, true_data_zmat, max_minutes = create_true(raw_concatenated_data, truetime, TR)
    all_data_dict, corr_value_list = get_corrs(true_data_zmat,test_data, max_minutes, all_data_dict, TR, subid)

#check how much true data you're using
seconds_true = math.floor(hour_array.shape[1] * TR)
hours_true = seconds_true // 3600
minutes_true = (seconds_true % 3600) // 60
seconds_true = seconds_true % 60
print('True data in time: {:02d}Hours {:02d}Min {:02d}Seconds'.format(hours_true, minutes_true, seconds_true))

#check how much data is in the test subset
seconds_test = math.floor(test_data.shape[1] * TR)
hours_test = seconds_test // 3600
minutes_test = (seconds_test % 3600) // 60
seconds_test = seconds_test % 60
print('Test data in time: {:02d}Hours {:02d}Min {:02d}Seconds'.format(hours_test, minutes_test, seconds_test))

In [ ]:
# Step 4.3: Clean and plot correlation values. 
all_data_dict = clean_dict(all_data_dict)
plot_final(all_data_dict, outdir, subid, truetime, fd_thresh, rands, num_of_sessions, subids)

### Now that we've demonstrated the reliability calculations from our default settings, you can adjust the following parameters and compare the figures generated: 
#####
##### 1. Adjust the variable **truetime** defined at the top of cell 7. We would suggest changing this to 5-15 minutes, which is the amount of data that group average studies typically collect.
#####
##### 2. Adjust the variable **num_of_sessions**. Make sure the truetime variable is not greater than total censored data in time (output of cell 5). 
#####
##### 3. You can also test how changing the number of iterations run affects the reliability curves.
#####
#### **We will take 10 minutes or so that everyone can test how changing these parameters affect the reliability estimates.**